# Load an Existing Chroma Database

This notebook reconnects to the persisted Chroma collection created in the CRUD notebook and reads its contents.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

## 1. Rebuild the Same Configuration

In [2]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

WindowsPath('d:/rag-vector-stores')

In [3]:
dotenv_path = project_root / ".env"
load_dotenv(dotenv_path=dotenv_path)

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Please add your OPENAI_API_KEY to the .env file before running this notebook.")

print(f"Loaded environment from: {dotenv_path}")

Loaded environment from: d:\rag-vector-stores\.env


In [12]:
collection_name = "demo_2"
persist_directory = project_root / "notebooks" / "chroma_langchain_db"

print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

Collection name: demo_2
Persist directory: d:\rag-vector-stores\notebooks\chroma_langchain_db


In [13]:
# Use the same embedding model that was used to build the store.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=str(persist_directory),
)

print("Connected to the existing Chroma collection.")

Connected to the existing Chroma collection.


## 2. Add Small Display Helpers

In [6]:
def preview_text(text, limit=80):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_stored_documents(records):
    """Print stored Chroma records in a readable format."""
    ids = records.get("ids", [])
    documents = records.get("documents", [])
    metadatas = records.get("metadatas", [])

    print(f"Total documents in collection: {len(ids)}")
    print()

    for index, (doc_id, document_text, metadata) in enumerate(zip(ids, documents, metadatas), start=1):
        print(f"{index}. id={doc_id}")
        print(f"   topic={metadata.get('topic')} | doc_number={metadata.get('doc_number')}")
        print(f"   content={preview_text(document_text)}")
        print()

## 3. Fetch and Inspect the Stored Records

In [14]:
stored_records = vector_store.get(include=["embeddings", "metadatas", "documents"])
stored_records.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

In [16]:
stored_records["embeddings"].shape

(8, 1536)

In [17]:
print_stored_documents(stored_records)

Total documents in collection: 8

1. id=73a1c0e2-e6d2-40c2-a128-6d244087f245
   topic=AI | doc_number=1
   content=Artificial intelligence helps machines perform tasks that usually need human rea...

2. id=f8efa688-bc5a-4a32-a2e6-c7d1806b0a28
   topic=AI | doc_number=2
   content=AI systems can analyze patterns in data to support predictions and automation.

3. id=ff63b73b-3685-4373-840b-ce0ed60b448c
   topic=AI | doc_number=3
   content=Responsible AI development includes fairness, transparency, and safety checks.

4. id=c07ea090-cbcb-45cf-a155-32069b5c40d1
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language m...

5. id=65baf5b7-0aaf-4159-9cd5-cd9fc5c3e504
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model ge...

6. id=3804cd92-b364-411d-a662-9cfc53650cd4
   topic=RAG | doc_number=6
   content=Vector stores are important in RAG because they make semantic 